# 01. Build patient_index (AD cases + non-AD controls)

Produces `output/patient_index.parquet` containing 19,858 AD cases
(from `ad_cohort.parquet`) and 19,858 non-AD controls sampled from
the OMOP person table with strict dementia exclusion.

Schema of output:

| column | meaning |
|---|---|
| `row_id` | 1..39716, used downstream to align X/Y matrices |
| `person_id` | OMOP person identifier |
| `index_datetime` | for AD cases: first AD code date; for controls: a borrowed date from the case index distribution (random pairing) |
| `birth_datetime` | from `person.birth_datetime` |
| `AGE_AT_INDEX` | years between birth and index |
| `group` | `'AD'` or `'non_AD'` |

Design choices (locked):

- **No washout.** This is a feature-characterization task, not a prediction task.
- **Controls are age-restricted to >=40 at borrowed index date**, matching the AD cohort's lower bound (the AD cohort in `ad_cohort.parquet` was defined with the same age >=40 cutoff).
- **No fine-grained demographic matching.** Controls are a simple random sample (post age filter) from the strict-non-dementia pool, with `index_datetime` borrowed from cases so calendar-time distributions match.
- **Strict control definition.** Any person with any of G30.*, F02.8x, F03.*, G31.83, G31.84, G31.09, F01.*, 331.0, 290.*, 294.1* is excluded from the control pool.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb

DATA_PATH = '/home/jupyter/2835794-data'
COHORT_PATH = 'ad_cohort.parquet'
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

RNG_SEED = 42
N_CONTROLS = 19858
MIN_AGE = 40           # matches the AD cohort lower bound (set in ADCohort.ipynb)
LOOKBACK_DAYS = 365    # recorded here for traceability; used in notebook 02

con = duckdb.connect()
print('DuckDB version:', duckdb.__version__)


## 1. Load AD cohort as the case half

In [ ]:
ad = pd.read_parquet(COHORT_PATH)
print('AD cohort:', ad.shape)

cases = (
    ad[['person_id', 'INDEX_DATE', 'BIRTH_DATE', 'AGE_AT_INDEX']]
    .rename(columns={'INDEX_DATE': 'index_datetime'})
    .assign(group='AD')
)
cases['index_datetime'] = pd.to_datetime(cases['index_datetime'])
cases['birth_datetime'] = pd.to_datetime(cases['BIRTH_DATE'])
cases = cases.drop(columns=['BIRTH_DATE'])
print('Cases:', len(cases))
cases.head()


## 2. Identify dementia-coded persons to exclude from the control pool

Strict definition: any person ever coded with any dementia-related ICD is removed.


In [ ]:
# Excluded codes:
#   ICD-10  G30.*    Alzheimer's disease (all subtypes)
#   ICD-10  F02.8x   Dementia in other diseases classified elsewhere
#   ICD-10  F03.*    Unspecified dementia
#   ICD-10  G31.83   Dementia with Lewy bodies
#   ICD-10  G31.84   Mild cognitive impairment, so stated
#   ICD-10  G31.09   Other frontotemporal dementia
#   ICD-10  F01.*    Vascular dementia
#   ICD-9   331.0    Alzheimer's disease (legacy)
#   ICD-9   290.*    Senile/presenile dementia
#   ICD-9   294.1*   Dementia in conditions classified elsewhere

EXCLUSION_PATTERNS_SQL = '''
       cond.condition_source_value LIKE 'G30%'
    OR cond.condition_source_value LIKE 'F02.8%'
    OR cond.condition_source_value LIKE 'F03%'
    OR cond.condition_source_value LIKE 'G31.83%'
    OR cond.condition_source_value LIKE 'G31.84%'
    OR cond.condition_source_value LIKE 'G31.09%'
    OR cond.condition_source_value LIKE 'F01%'
    OR cond.condition_source_value = '331.0'
    OR cond.condition_source_value LIKE '290%'
    OR cond.condition_source_value LIKE '294.1%'
'''

excluded_persons = con.sql(f'''
SELECT DISTINCT person_id
FROM read_parquet('{DATA_PATH}/condition_occurrence/*.parquet') cond
WHERE {EXCLUSION_PATTERNS_SQL}
''').to_df()

print(f'Persons excluded from control pool: {len(excluded_persons):,}')


## 3. Build the control candidate pool

A control candidate is any person who:
- appears in `person.parquet`
- is NOT in the excluded set above
- has at least one `condition_occurrence` record (proves real EHR contact)


In [ ]:
candidate_pool = con.sql(f'''
WITH all_persons AS (
    SELECT DISTINCT person_id, birth_datetime
    FROM read_parquet('{DATA_PATH}/person/*.parquet')
),
persons_with_any_condition AS (
    SELECT DISTINCT person_id
    FROM read_parquet('{DATA_PATH}/condition_occurrence/*.parquet')
)
SELECT p.person_id, p.birth_datetime
FROM all_persons p
WHERE p.person_id IN (SELECT person_id FROM persons_with_any_condition)
  AND p.person_id NOT IN (SELECT person_id FROM excluded_persons)
''').to_df()

print(f'Control candidate pool size: {len(candidate_pool):,}')


## 4. Sample 19,858 controls (age >=40, borrowed index_datetime from cases)

- Each candidate is pre-assigned a borrowed `index_datetime` from a random case
- Only candidates with `AGE_AT_INDEX >= MIN_AGE (40)` at that borrowed date are eligible
- Sample N_CONTROLS without replacement from the eligible pool
- This matches the AD cohort's age >=40 lower bound, giving an aging control cohort


In [ ]:
rng = np.random.default_rng(RNG_SEED)

candidate_pool = candidate_pool.copy()
candidate_pool['birth_datetime'] = pd.to_datetime(candidate_pool['birth_datetime'])

# Step 1: pre-assign every candidate a borrowed index_datetime (with replacement here;
# the final N_CONTROLS persons are unique, but dates may repeat across them)
random_dates = (
    cases['index_datetime']
    .sample(n=len(candidate_pool), replace=True, random_state=RNG_SEED)
    .reset_index(drop=True)
)
candidate_pool = candidate_pool.reset_index(drop=True)
candidate_pool['index_datetime'] = random_dates
candidate_pool['AGE_AT_INDEX'] = (
    (candidate_pool['index_datetime'] - candidate_pool['birth_datetime']).dt.days / 365.25
)

# Step 2: keep only candidates who are >= MIN_AGE at their borrowed index
eligible = candidate_pool[candidate_pool['AGE_AT_INDEX'] >= MIN_AGE].reset_index(drop=True)
print(f'Candidates with age >= {MIN_AGE} at borrowed index: {len(eligible):,}')

if len(eligible) < N_CONTROLS:
    raise ValueError(
        f'Eligible pool ({len(eligible)}) smaller than N_CONTROLS ({N_CONTROLS}).'
    )

# Step 3: sample N_CONTROLS without replacement
sampled_idx = rng.choice(len(eligible), size=N_CONTROLS, replace=False)
controls = eligible.iloc[sampled_idx].reset_index(drop=True).copy()
controls['group'] = 'non_AD'
controls = controls[['person_id', 'index_datetime', 'birth_datetime', 'AGE_AT_INDEX', 'group']]

print(f'Final controls sampled: {len(controls):,}')
print()
print('Case vs control age summary:')
print(pd.DataFrame({
    'case': cases['AGE_AT_INDEX'].describe(),
    'control': controls['AGE_AT_INDEX'].describe(),
}))


## 5. Combine cases + controls into one `patient_index` table

In [ ]:
patient_index = pd.concat(
    [cases[['person_id', 'index_datetime', 'birth_datetime', 'AGE_AT_INDEX', 'group']],
     controls[['person_id', 'index_datetime', 'birth_datetime', 'AGE_AT_INDEX', 'group']]],
    ignore_index=True
)
patient_index['row_id'] = np.arange(1, len(patient_index) + 1)
patient_index = patient_index[['row_id', 'person_id', 'index_datetime',
                                'birth_datetime', 'AGE_AT_INDEX', 'group']]

print('Combined patient_index shape:', patient_index.shape)
print()
print('Group counts:')
print(patient_index['group'].value_counts())
print()
print('Age distribution by group:')
print(patient_index.groupby('group')['AGE_AT_INDEX'].describe())
print()
print('Index date range by group:')
print(patient_index.groupby('group')['index_datetime'].agg(['min', 'max']))


## 6. Save

In [ ]:
out_path = OUTPUT_DIR / 'patient_index.parquet'
patient_index.to_parquet(out_path, index=False)
print(f'Saved: {out_path}  ({len(patient_index):,} rows)')

excluded_persons.to_parquet(OUTPUT_DIR / 'control_excluded_persons.parquet', index=False)
print(f'Saved exclusion list: {len(excluded_persons):,} persons')
